In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the data
data = pd.read_csv("/kaggle/input/datasets/tharsthanu/coffee-sales/index.csv")

print("First 5 rows:")
print(data.head())


In [ ]:
# Convert date columns
data['date'] = pd.to_datetime(data['date'])
data['datetime'] = pd.to_datetime(data['datetime'])

In [ ]:
# Keep missing card as NaN-these are cash transactions
print(f"\nMissing values before handling:\n{data.isnull().sum()}")

In [ ]:
# Fill missing card values with 'cash' (since they are cash transactions)
data['card'] = data['card'].fillna('cash')

print(f"\nMissing values after handling:\n{data.isnull().sum()}")

In [ ]:
# Extract time features
data['month'] = data['date'].dt.to_period('M').astype(str)
data['day_of_month'] = data['date'].dt.day
data['weekday'] = data['date'].dt.day_name()
data['weekday_num'] = data['date'].dt.dayofweek  # Monday=0, Sunday=6
data['hour'] = data['datetime'].dt.hour
data['week'] = data['date'].dt.isocalendar().week.astype(int)
data['is_weekend'] = (data['weekday_num'] >= 5).astype(int)

print("\nData after feature engineering:")
print(data.head())


In [ ]:
#DATA OVERVIEW
print(f"\nShape of dataset: {data.shape}")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")

print("\nCoffee types count:")
print(data['coffee_name'].value_counts())

print("\nPayment type count:")
print(data['cash_type'].value_counts())

print("\nSummary statistics:")
print(data[['money', 'day_of_month', 'hour', 'week']].describe())


In [ ]:
# Daily total sales
daily_sales = data.groupby('date')['money'].sum().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(daily_sales['date'], daily_sales['money'], linewidth=1)
plt.title("Daily Sales Trend (March - July 2024)")
plt.xlabel("Date")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Monthly total sales
monthly_sales = data.groupby('month')['money'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='month', y='money', data=monthly_sales, color='steelblue')
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sales by weekday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_sales = data.groupby('weekday')['money'].sum().reindex(weekday_order).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='weekday', y='money', data=weekly_sales, palette='viridis')
plt.title("Sales by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Sales by hour
hourly_sales = data.groupby('hour')['money'].sum().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(x='hour', y='money', data=hourly_sales, palette='coolwarm')
plt.title("Sales by Hour of Day")
plt.xlabel("Hour (24h)")
plt.ylabel("Total Sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Product popularity
plt.figure(figsize=(10, 6))
product_counts = data['coffee_name'].value_counts()
sns.barplot(y=product_counts.index, x=product_counts.values, palette='Set2')
plt.title("Number of Sales by Coffee Type")
plt.xlabel("Number of Transactions")
plt.ylabel("Coffee Name")
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by product
coffee_revenue = data.groupby('coffee_name')['money'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='money', y='coffee_name', data=coffee_revenue, palette='Set3')
plt.title("Revenue by Coffee Type")
plt.xlabel("Revenue ($)")
plt.ylabel("Coffee Name")
plt.tight_layout()
plt.show()

In [ ]:
# Payment method distribution
payment_counts = data['cash_type'].value_counts(normalize=True) * 100
print("\nPayment method percentage:")
print(payment_counts)

In [ ]:
# SALES PREDICTION
# Aggregate daily sales
daily_sales = data.groupby('date').agg({
    'money': 'sum',
    'datetime': 'count'
}).rename(columns={'datetime': 'transaction_count'}).reset_index()

In [ ]:
# Create time-based features
daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['day_of_month'] = daily_sales['date'].dt.day
daily_sales['week_of_year'] = daily_sales['date'].dt.isocalendar().week

In [ ]:
# Create lag features (previous days sales)
daily_sales['prev_day_sales'] = daily_sales['money'].shift(1)
daily_sales['prev_2day_sales'] = daily_sales['money'].shift(2)
daily_sales['prev_3day_sales'] = daily_sales['money'].shift(3)
daily_sales['prev_week_sales'] = daily_sales['money'].shift(7)

In [ ]:
# Create rolling averages
daily_sales['rolling_3day_avg'] = daily_sales['money'].rolling(window=3, min_periods=1).mean()
daily_sales['rolling_7day_avg'] = daily_sales['money'].rolling(window=7, min_periods=1).mean()

In [ ]:
# Drop rows with NaN values from lag features
daily_sales_clean = daily_sales.dropna().reset_index(drop=True)

print(f"\nDaily sales data shape: {daily_sales_clean.shape}")
print(f"Date range for modeling: {daily_sales_clean['date'].min()} to {daily_sales_clean['date'].max()}")

In [ ]:
# Features for modeling
feature_columns = [
    'day_of_week', 'month', 'day_of_month', 'week_of_year',
    'prev_day_sales', 'prev_2day_sales', 'prev_3day_sales', 'prev_week_sales',
    'rolling_3day_avg', 'rolling_7day_avg', 'transaction_count'
]

X = daily_sales_clean[feature_columns]
y = daily_sales_clean['money']

In [ ]:
# Chronological split (80% train, 20% test - keeping order)
split_idx = int(len(daily_sales_clean) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nTraining set: {X_train.shape[0]} days ({daily_sales_clean['date'].iloc[split_idx-1]})")
print(f"Test set: {X_test.shape[0]} days ({daily_sales_clean['date'].iloc[split_idx]} to {daily_sales_clean['date'].iloc[-1]})")

In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Mean Squared Error: {mse_lr:.2f}")
print(f"Root Mean Squared Error: {rmse_lr:.2f}")
print(f"Mean Absolute Error: {mae_lr:.2f}")
print(f"R² Score: {r2_lr:.4f}")

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Mean Squared Error: {mse_rf:.2f}")
print(f"Root Mean Squared Error: {rmse_rf:.2f}")
print(f"Mean Absolute Error: {mae_rf:.2f}")
print(f"R² Score: {r2_rf:.4f}")

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance (Random Forest):")
print(feature_importance)

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(14, 6))
test_dates = daily_sales_clean['date'].iloc[split_idx:]

plt.plot(test_dates, y_test.values, label='Actual Sales', marker='o', linewidth=2, markersize=4)
plt.plot(test_dates, y_pred_rf, label='Random Forest Predictions', marker='s', linewidth=2, markersize=4, alpha=0.7)
plt.plot(test_dates, y_pred_lr, label='Linear Regression Predictions', marker='^', linewidth=2, markersize=4, alpha=0.7)
plt.title('Actual vs Predicted Sales (Test Set)')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#FUTURE PREDICTIONS

# Get the last row to use as base for predictions
last_row = daily_sales_clean.iloc[-1:].copy()
last_date = daily_sales_clean['date'].max()

In [ ]:
# Function to predict next day iteratively
def predict_future_days(model, last_data, feature_cols, n_days=7):
    predictions = []
    current_data = last_data.copy()
    current_date = last_date
    
    for i in range(n_days):
        current_date += pd.Timedelta(days=1)
        
        # Update date-based features
        current_data['day_of_week'] = current_date.dayofweek
        current_data['month'] = current_date.month
        current_data['day_of_month'] = current_date.day
        current_data['week_of_year'] = current_date.isocalendar().week
        
        # Make prediction
        pred = model.predict(current_data[feature_cols])[0]
        predictions.append(pred)
        
        # Update lag features for next iteration
        if i < n_days - 1:
            current_data['prev_day_sales'] = pred
            current_data['prev_2day_sales'] = current_data['prev_day_sales'].values[0]
            current_data['prev_3day_sales'] = current_data['prev_2day_sales'].values[0]
            current_data['prev_week_sales'] = current_data['prev_day_sales'].values[0]
            current_data['rolling_3day_avg'] = (current_data['prev_3day_sales'].values[0] + 
                                                  current_data['prev_2day_sales'].values[0] + pred) / 3
            current_data['rolling_7day_avg'] = current_data['rolling_3day_avg'].values[0]
    
    return predictions

In [ ]:
# Predict next 7 days using Random Forest (better model)
next_7_days = predict_future_days(rf_model, last_row, feature_columns, 7)

future_dates = [last_date + pd.Timedelta(days=i+1) for i in range(7)]
future_df = pd.DataFrame({
    'date': future_dates,
    'predicted_sales': next_7_days
})

print("\nNext 7 Days Sales Prediction (Random Forest):")
print(future_df.to_string(index=False))
print(f"\nTotal predicted sales for next week: ${sum(next_7_days):.2f}")


In [ ]:
# Predict next 30 days
next_30_days = predict_future_days(rf_model, last_row, feature_columns, 30)
future_30_dates = [last_date + pd.Timedelta(days=i+1) for i in range(30)]

plt.figure(figsize=(14, 6))
plt.plot(future_30_dates, next_30_days, marker='o', linewidth=2, markersize=4)
plt.title('Next 30 Days Sales Prediction (Random Forest)')
plt.xlabel('Date')
plt.ylabel('Predicted Sales ($)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#CUSTOMER PURCHASE ANALYSIS
# Customer data (excluding cash transactions for customer analysis)
customer_data = data[data['card'] != 'cash'].copy()

print(f"\nNumber of unique customers: {customer_data['card'].nunique()}")

In [ ]:
# Top customers by purchase count
top_customers_count = customer_data['card'].value_counts().head(10).reset_index()
top_customers_count.columns = ['customer_id', 'purchase_count']

print("\nTop 10 customers by purchase count:")
print(top_customers_count.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='purchase_count', y='customer_id', data=top_customers_count, palette='Blues_d')
plt.title("Top 10 Customers by Number of Purchases")
plt.xlabel("Purchase Count")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()


In [ ]:
# Top customers by total spending
top_customers_spending = customer_data.groupby('card')['money'].sum().sort_values(ascending=False).head(10).reset_index()
top_customers_spending.columns = ['customer_id', 'total_spending']

print("\nTop 10 customers by total spending:")
print(top_customers_spending.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='total_spending', y='customer_id', data=top_customers_spending, palette='Greens_d')
plt.title("Top 10 Customers by Total Spending")
plt.xlabel("Total Spending ($)")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()

In [ ]:
# Analyze specific customer (most frequent)
specific_customer = top_customers_count['customer_id'].iloc[0]
specific_customer_data = customer_data[customer_data['card'] == specific_customer]


print(f"CUSTOMER DETAILS: {specific_customer}")

print(f"\nTotal purchases: {len(specific_customer_data)}")
print(f"Total spending: ${specific_customer_data['money'].sum():.2f}")
print(f"Average spending per purchase: ${specific_customer_data['money'].mean():.2f}")
print(f"Favorite payment method: {specific_customer_data['cash_type'].mode().iloc[0]}")

print("\nPurchase history (last 10 transactions):")
print(specific_customer_data[['date', 'datetime', 'coffee_name', 'money']].tail(10).to_string(index=False))

In [ ]:
# Favorite products of this customer
customer_favorites = specific_customer_data['coffee_name'].value_counts().reset_index()
customer_favorites.columns = ['coffee_name', 'count']

print("\nFavorite products:")
print(customer_favorites.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='count', y='coffee_name', data=customer_favorites, palette='Oranges_d')
plt.title(f"Favorite Products of Customer {specific_customer}")
plt.xlabel("Purchase Count")
plt.ylabel("Coffee Name")
plt.tight_layout()
plt.show()

In [ ]:
#FINAL INSIGHTS & RECOMMENDATIONS

print("FINAL INSIGHTS & RECOMMENDATIONS")

print(f"\n OVERALL METRICS:")
print(f"   • Total Revenue: ${data['money'].sum():.2f}")
print(f"   • Total Transactions: {len(data)}")
print(f"   • Average Transaction Value: ${data['money'].mean():.2f}")
print(f"   • Date Range: {data['date'].min().date()} to {data['date'].max().date()}")

print(f"\n TOP PERFORMING PRODUCTS:")
top_products = data['coffee_name'].value_counts().head(3)
for product, count in top_products.items():
    revenue = data[data['coffee_name'] == product]['money'].sum()
    print(f"   • {product}: {count} sales, ${revenue:.2f} revenue")

print(f"\n SALES PATTERNS:")
print(f"   • Best Day: {weekly_sales.loc[weekly_sales['money'].idxmax(), 'weekday']}")
print(f"   • Best Hour: {hourly_sales.loc[hourly_sales['money'].idxmax(), 'hour']}:00")
print(f"   • Weekend vs Weekday: Weekend sales represent {(data[data['is_weekend']==1]['money'].sum() / data['money'].sum() * 100):.1f}% of total")

print(f"\n CUSTOMER INSIGHTS:")
print(f"   • Unique Customers: {customer_data['card'].nunique()}")
print(f"   • Top Customer: {specific_customer} with {len(specific_customer_data)} purchases")
print(f"   • Customer Retention Opportunity: {len(customer_data[customer_data.groupby('card')['card'].transform('size') > 1])} repeat customers")

print(f"\n PAYMENT PREFERENCES:")
print(f"   • Card: {payment_counts['card']:.1f}%")
print(f"   • Cash: {payment_counts['cash']:.1f}%")
